# Leakage Audit and Donor-Grouped Evaluation Folds

## Objective

Before comparing tabular-only and hybrid (tabular + image) models, verify that:

1. None of the 24 tabular features directly encodes the target grade.
2. Every feature is legitimately available at the intended prediction time.
3. Evaluation folds are independent at the donor level.

This notebook also creates the donor-grouped folds used by all later model comparisons.

## Prediction Scenarios

| Scenario | Description | Slit-lamp findings allowed? |
|---|---|---|
| **Primary: expert decision support** | The specialist records findings during tissue evaluation; the AI helps assign, verify and explain the grade | Yes |
| Secondary: pre-examination prediction | The AI predicts the grade before the specialist examines the tissue | No |

**Decision:** The primary task is AI-assisted expert grading during tissue evaluation. Slit-lamp findings recorded during that evaluation stay in the primary feature set. Scenarios with missing clinical information are tested separately as a secondary experiment and are not mixed into the primary model.

## Research Question

> Can image-derived information improve donor cornea tissue grading beyond existing clinical and endothelial morphology measurements, particularly when some measurements are unavailable?

In [1]:
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path("../../scripts").resolve()))
import modeling_setup as ms

labelled = ms.load_labelled_data()
X = labelled[ms.ALL_FEATURES]
y = labelled[ms.TARGET]

print("Labelled tissues:", len(labelled))
print("Grade counts:", y.value_counts().to_dict())
print("Features:", len(ms.ALL_FEATURES))

Labelled tissues: 1036
Grade counts: {'A+': 545, 'A': 319, 'B': 172}
Features: 24


## 1. Feature Provenance

Each feature is traced to the raw column it was derived from in `02_data_cleaning.ipynb`, and checked for availability in each scenario.

In [2]:
source_column = {f: "Konan specular report" for f in ms.NUMERIC_FEATURES}
for f in ms.CLINICAL_FEATURES:
    prefix = f.split("_")[0]
    source_column[f] = {
        "Stroma": "Stroma_Findings",
        "Epi": "Epithelium_Findings",
        "Endo": "Endothelium_Findings",
        "Descemet": "Descemet_Findings",
    }[prefix]

provenance = pd.DataFrame({
    "Feature": ms.ALL_FEATURES,
    "Group": [
        "Endothelial morphology" if f in ms.NUMERIC_FEATURES
        else "Slit-lamp finding" for f in ms.ALL_FEATURES
    ],
    "Source": [source_column[f] for f in ms.ALL_FEATURES],
    "Missing_%": (X.isna().mean() * 100).round(1).to_numpy(),
    "Primary_scenario": "Yes",
    "Pre_examination": [
        "Yes" if f in ms.NUMERIC_FEATURES else "No"
        for f in ms.ALL_FEATURES
    ],
})

print(provenance.to_string(index=False))

                      Feature                  Group                Source  Missing_% Primary_scenario Pre_examination
         ECD_CD_cells_per_mm2 Endothelial morphology Konan specular report        9.3              Yes             Yes
                           CV Endothelial morphology Konan specular report        9.3              Yes             Yes
                  HEX_percent Endothelial morphology Konan specular report        9.3              Yes             Yes
                           SD Endothelial morphology Konan specular report        9.3              Yes             Yes
             Total_Cell_Count Endothelial morphology Konan specular report        9.3              Yes             Yes
               Total_Area_um2 Endothelial morphology Konan specular report        9.3              Yes             Yes
              Stroma_Is_Clear      Slit-lamp finding       Stroma_Findings       44.7              Yes              No
            Stroma_Is_Compact      Slit-lamp fin

## 2. Direct Encoding of the Grade

Free-text source columns are searched for grade labels, suitability decisions or planned procedures. Any such mention in a source column would let a parsed feature encode the label.

In [3]:
label_pattern = re.compile(
    r"\b(grade|A\+|optical|tectonic|suitab\w*|unsuitab\w*|discard\w*|"
    r"reject\w*|PK|DSAEK|DMEK|EK)\b",
    flags=re.IGNORECASE,
)

text_columns = [
    "Epithelium_Findings",
    "Stroma_Findings",
    "Descemet_Findings",
    "Endothelium_Findings",
    "Additional_Findings",
    "Konan_Comments",
]

rows = []
for column in text_columns:
    text = labelled[column].dropna().astype(str)
    hits = text[text.str.contains(label_pattern)]
    rows.append({
        "Column": column,
        "Used_for_features": column in set(source_column.values()),
        "Non_empty": len(text),
        "Label_mentions": len(hits),
        "Example": hits.iloc[0][:70] if len(hits) else "",
    })

print(pd.DataFrame(rows).to_string(index=False))

post_decision = {
    "Overall_Donor_Quality", "Transplant_Suitability", "Suitability_Reason",
    "Tissue_Outcome", "Requested_Grade", "Grade_Request_Met",
    "Intended_Use", "Additional_Findings", "Review_Notes",
}
print("\nPost-decision columns used as features:",
      sorted(post_decision & set(ms.ALL_FEATURES)) or "none")

              Column  Used_for_features  Non_empty  Label_mentions                                                                Example
 Epithelium_Findings               True        572               0                                                                       
     Stroma_Findings               True        573               0                                                                       
   Descemet_Findings               True        571               0                                                                       
Endothelium_Findings               True        561               0                                                                       
 Additional_Findings              False        572             256 Release form: Grade A+, irregular cut; rim 2.5-3 mm; C-S size 18 mm; c
      Konan_Comments              False        953               0                                                                       

Post-decision columns used as fea

## 3. Single-Feature Predictive Strength

A single feature that predicts the grade almost perfectly would suggest it encodes the label. Each feature is evaluated alone with a Random Forest and 5-fold stratified CV.

In [4]:
def single_feature_pipeline(feature):
    return Pipeline([
        ("preprocessor", ms.make_preprocessor(
            numeric=[feature] if feature in ms.NUMERIC_FEATURES else [],
            binary=[feature] if feature in ms.CLINICAL_BINARY_FEATURES else [],
            categorical=(
                [feature] if feature in ms.CLINICAL_CATEGORICAL_FEATURES else []
            ),
        )),
        ("classifier", RandomForestClassifier(
            n_estimators=200, class_weight="balanced",
            random_state=ms.RANDOM_STATE, n_jobs=-1,
        )),
    ])

single_feature = pd.DataFrame([
    {
        "Feature": feature,
        "Macro_F1": cross_val_score(
            single_feature_pipeline(feature), X[[feature]], y,
            cv=ms.make_cv(), scoring="f1_macro",
        ).mean(),
    }
    for feature in ms.ALL_FEATURES
]).sort_values("Macro_F1", ascending=False)

print(single_feature.round(3).to_string(index=False))

                      Feature  Macro_F1
         ECD_CD_cells_per_mm2     0.771
               Total_Area_um2     0.572
                           SD     0.554
             Total_Cell_Count     0.382
                  HEX_percent     0.366
                           CV     0.363
                Epi_Has_Arcus     0.330
             Epi_Has_Exposure     0.303
Descemet_Fold_Relative_Number     0.302
    Endo_Stress_Line_Location     0.299
              Stroma_Has_Scar     0.266
        Epi_Exposure_Severity     0.257
       Descemet_Fold_Severity     0.250
             Stroma_Has_Arcus     0.249
              Stroma_Is_Clear     0.248
            Stroma_Is_Compact     0.247
             Stroma_Has_Edema     0.247
               Epi_Has_Defect     0.247
        Endo_Has_Cell_Dropout     0.246
                 Epi_Has_Haze     0.246
                 Endo_Has_RBC     0.246
        Endo_Has_Stress_Lines     0.244
        Stroma_Arcus_Severity     0.242
           Descemet_Has_Folds     0.235


## 4. Is Missingness Informative?

If missing values were associated with the grade, a model could learn from the absence of data rather than the data itself, and the complete-versus-incomplete comparison planned for the secondary experiment would be confounded.

In [5]:
clinical_missing = X[ms.CLINICAL_BINARY_FEATURES].isna().all(axis=1)
numeric_missing = X[ms.NUMERIC_FEATURES].isna().all(axis=1)

for name, mask in [
    ("All slit-lamp findings missing", clinical_missing),
    ("All Konan measurements missing", numeric_missing),
]:
    print(name)
    print(pd.crosstab(mask, y, normalize="index").round(3), "\n")

missing_only = cross_val_score(
    RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        random_state=ms.RANDOM_STATE,
    ),
    X.isna().astype(int), y, cv=ms.make_cv(), scoring="f1_macro",
)
print("Macro F1 from missingness indicators only:",
      round(missing_only.mean(), 3))

All slit-lamp findings missing
Tissue_Grade_Clean      A     A+      B
row_0                                  
False               0.349  0.504  0.147
True                0.257  0.553  0.190 

All Konan measurements missing
Tissue_Grade_Clean      A     A+      B
row_0                                  
False               0.299  0.537  0.164
True                0.396  0.417  0.188 



Macro F1 from missingness indicators only: 0.272


## 5. ECD Source Consistency

The Konan ECD used as a feature is compared with the ECD written on the eye-bank form that the specialist used when grading.

In [6]:
ecd = labelled[["ECD_CD_cells_per_mm2", "Form_ECD_cells_per_mm2"]].apply(
    pd.to_numeric, errors="coerce"
).dropna()

print("Tissues with both values:", len(ecd))
print("Correlation:", round(ecd.corr().iloc[0, 1], 3))
print("Exactly equal:", f"{(ecd.iloc[:, 0] == ecd.iloc[:, 1]).mean():.1%}")

Tissues with both values: 479
Correlation: 0.997
Exactly equal: 97.5%


## 6. Donor Independence

Fellow eyes from one donor receive consecutive tissue serial numbers. `ms.get_donor_groups` links consecutive opposite-eye tissues that share donor age and cause or time of death.

In [7]:
labelled["Donor_Group"] = ms.get_donor_groups(labelled)

group_sizes = labelled["Donor_Group"].value_counts()
print("Labelled tissues:", len(labelled))
print("Inferred donors:", labelled["Donor_Group"].nunique())
print("Group sizes:", group_sizes.value_counts().sort_index().to_dict())

pairs = labelled[labelled["Donor_Group"].isin(group_sizes[group_sizes == 2].index)]
pair_grades = pairs.groupby("Donor_Group")[ms.TARGET].agg(list)
same_grade = pair_grades.apply(lambda g: g[0] == g[1]).mean()

class_share = y.value_counts(normalize=True)
chance_agreement = (class_share ** 2).sum()

print(f"\nFellow eyes with the same grade: {same_grade:.1%}")
print(f"Expected by chance: {chance_agreement:.1%}")

# Consistency of donor details within inferred pairs, without printing them.
def pair_agreement(column):
    values = pairs.groupby("Donor_Group")[column].agg(list)
    both = values[values.apply(lambda v: pd.notna(v[0]) and pd.notna(v[1]))]
    return pd.Series({
        "Pairs with both values": len(both),
        "Identical": both.apply(lambda v: v[0] == v[1]).mean(),
    })

print()
print("Agreement of donor details within inferred pairs:")
print(pd.DataFrame({
    column: pair_agreement(column)
    for column in ["Donor_Age", "Primary_COD", "Death_Date_Time"]
}).T.round(3).to_string())

Labelled tissues: 1036
Inferred donors: 778
Group sizes: {1: 520, 2: 258}

Fellow eyes with the same grade: 77.9%
Expected by chance: 39.9%

Agreement of donor details within inferred pairs:
                 Pairs with both values  Identical
Donor_Age                         258.0       1.00
Primary_COD                       258.0       0.88
Death_Date_Time                   208.0       1.00


In [8]:
def pairs_split_apart(split_df, id_column, split_column, id_map=None):
    ids = labelled["Tissue_ID"]
    if id_map is not None:
        ids = ids.map(id_map)
    assignment = ids.map(dict(zip(split_df[id_column], split_df[split_column])))
    per_donor = (
        pd.DataFrame({"donor": labelled["Donor_Group"], "split": assignment})
        .dropna()
        .groupby("donor")["split"]
        .agg(["size", "nunique"])
    )
    paired = per_donor[per_donor["size"] > 1]
    return len(paired), int((paired["nunique"] > 1).sum())

holdout = pd.read_csv(ms.SPLIT_PATH)
shared = pd.read_csv(
    ms.PROCESSED_DIR.parent.parent
    / "image/data/processed/splits/multimodal_shared_split.csv"
)

def normalize_id(value):
    return " ".join(str(value).split()).replace("NEB- ", "NEB-").casefold()

for name, split_df, id_column, id_map in [
    ("80/20 tabular holdout", holdout, "Tissue_ID", None),
    ("Shared image/tabular split", shared, "Normalized_ID", normalize_id),
]:
    n_pairs, n_apart = pairs_split_apart(split_df, id_column, "Split", id_map)
    print(f"{name}: {n_apart} of {n_pairs} donor pairs split across partitions")

80/20 tabular holdout: 78 of 258 donor pairs split across partitions
Shared image/tabular split: 108 of 239 donor pairs split across partitions


### Effect on the tabular estimate

The tuned Random Forest from `05_model_selection_and_evaluation.ipynb` is evaluated with ordinary stratified folds and with donor-grouped folds, each repeated five times.

In [9]:
def tuned_rf():
    return Pipeline([
        ("preprocessor", ms.make_preprocessor()),
        ("classifier", RandomForestClassifier(
            n_estimators=300, max_depth=15, min_samples_split=5,
            min_samples_leaf=1, max_features="sqrt",
            class_weight="balanced", random_state=ms.RANDOM_STATE, n_jobs=-1,
        )),
    ])

scores = {"Stratified (ignores donors)": [], "Donor-grouped": []}
groups = labelled["Donor_Group"]

for repeat in range(ms.N_REPEATS):
    seed = ms.RANDOM_STATE + repeat
    scores["Stratified (ignores donors)"] += list(cross_val_score(
        tuned_rf(), X, y, scoring="f1_macro",
        cv=StratifiedKFold(ms.N_SPLITS, shuffle=True, random_state=seed),
    ))
    scores["Donor-grouped"] += list(cross_val_score(
        tuned_rf(), X, y, groups=groups, scoring="f1_macro",
        cv=ms.make_group_cv(random_state=seed),
    ))

print(pd.DataFrame({
    name: {"Mean macro F1": np.mean(v), "SD": np.std(v), "Folds": len(v)}
    for name, v in scores.items()
}).T.round(4))

                             Mean macro F1      SD  Folds
Stratified (ignores donors)         0.8580  0.0150   25.0
Donor-grouped                       0.8437  0.0201   25.0


## 7. Donor-Grouped Folds for All Later Experiments

Five repeats of stratified donor-grouped 5-fold CV are created over all labelled tissues and saved to `data/processed/donor_grouped_folds.csv`. Tabular, image and hybrid models will all read this file, so their fold-level scores can be compared pairwise.

In [10]:
folds = ms.get_donor_folds(labelled)
fold_columns = [f"Fold_r{r}" for r in range(ms.N_REPEATS)]

# Each donor must sit in exactly one fold in every repeat.
for column in fold_columns:
    assert (folds.groupby("Donor_Group")[column].nunique() == 1).all(), column
print("Donor integrity: every donor is in a single fold in all repeats.")

check = folds.merge(labelled[["Tissue_ID", ms.TARGET]], on="Tissue_ID")
print("\nGrade distribution per fold (repeat 0):")
print(pd.crosstab(check["Fold_r0"], check[ms.TARGET])[ms.CLASSES])

Donor integrity: every donor is in a single fold in all repeats.

Grade distribution per fold (repeat 0):
Tissue_Grade_Clean   A+   A   B
Fold_r0                        
0                   109  64  35
1                   109  64  34
2                   109  63  35
3                   109  64  34
4                   109  64  34


In [11]:
manifest = pd.read_csv(
    ms.PROCESSED_DIR.parent.parent
    / "image/data/processed/clean_image_manifest.csv"
)
review_ids = {"NEB-2024-08-16259 R", "NEB-2025-09-17471 R"}
image_ids = {
    normalize_id(i) for i in manifest["Tissue_ID"]
} - {normalize_id(i) for i in review_ids}

image_cohort = check[check["Tissue_ID"].map(normalize_id).isin(image_ids)]
print("Labelled tissues with an image:", len(image_cohort))
print("\nImage-cohort grade distribution per fold (repeat 0):")
print(pd.crosstab(image_cohort["Fold_r0"], image_cohort[ms.TARGET])[ms.CLASSES])

Labelled tissues with an image: 1009

Image-cohort grade distribution per fold (repeat 0):
Tissue_Grade_Clean   A+   A   B
Fold_r0                        
0                   106  62  33
1                   108  64  34
2                   106  61  32
3                   105  64  34
4                   105  63  32


## Summary

### Leakage findings

| Check | Result | Verdict |
|---|---|---|
| Grade text in feature source columns | 0 mentions across the four findings columns | Clean |
| `Additional_Findings` | Contains release-form grades (256 of 572) | Already excluded in `02_data_cleaning`; never used for features |
| Post-decision columns among features | None | Clean |
| Strongest single feature | ECD, macro F1 0.771; all slit-lamp features alone 0.24–0.33 | No feature encodes the label |
| Missingness indicators alone | Macro F1 0.272 (near chance) | Missingness is not informative |
| Konan ECD vs eye-bank form ECD | r = 0.997, 97.5% identical | Feature matches what the specialist saw |

All 24 features are available in the primary scenario (expert decision support). Only the six Konan morphology features would be available in the pre-examination scenario.

### Donor independence

- 1,036 labelled tissues come from about 778 donors (258 fellow-eye pairs).
- Fellow eyes share a grade 77.9% of the time, against 39.9% expected by chance.
- Earlier splits separated fellow eyes: 78 of 258 pairs in the 80/20 tabular holdout and 108 of 239 pairs in the shared image/tabular split.
- Grouping by donor lowers the tuned Random Forest from 0.858 to 0.844 macro F1 (5×5 CV). Earlier tabular estimates were slightly optimistic. The effect on image models has not been measured and may be larger.

### Consequences for later experiments

1. All new comparisons use `donor_grouped_folds.csv` through `ms.get_donor_folds` and `ms.iter_donor_folds`. Tabular, image and hybrid models are evaluated on identical folds and compared pairwise across the 25 fold scores.
2. The earlier holdout and shared-split results remain baseline results only; they are not compared directly with donor-grouped results.
3. The primary model keeps all 24 features. Feature-masking and missing-clinical-information experiments are reported separately as the secondary scenario.
4. Donor pairing is inferred from consecutive serial numbers and donor details. It should be confirmed against the eye bank's donor register if one is available.

## Addendum (after notebook 08)

Missingness indicators do not predict the grade (Section 4), but slit-lamp missingness is not random with respect to the data: it coincides almost exactly with tissues from 2023, whose recorded grades follow a different pattern from other years. See `08_missing_information.ipynb`, Section 10. Results pooled across years should therefore also be reported by year.